In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

super_ai_engineer_ss_6_thai_language_image_captioning_path = kagglehub.competition_download('super-ai-engineer-ss-6-thai-language-image-captioning')

print('Data source import complete.')


In [ ]:
!pip install -U transformers accelerate datasets pillow pandas sentencepiece
!pip install "Pillow<11.0.0"

  Using cached pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
Using cached pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (7.1 MB)
  Attempting uninstall: pillow
    Found existing installation: pillow 10.4.0
    Uninstalling pillow-10.4.0:
      Successfully uninstalled pillow-10.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
  Using cached pillow-10.4.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (9.2 kB)
Using c

In [ ]:
import json
import torch
import os
import pandas as pd
from PIL import Image
from tqdm import tqdm
from datasets import Dataset
from transformers import (
    VisionEncoderDecoderModel,
    ViTModel,
    GPT2LMHeadModel,
    AutoImageProcessor,
    AutoTokenizer,
    AutoConfig,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

In [ ]:
def load_json_to_dataset(json_path, img_root_path=""):
    with open(json_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

    data_list = []
    for img_path, captions in raw_data.items():
        if img_path.startswith('coco'): continue

        cap = captions[0]
        parts = img_path.split('/')
        sub_path = "/".join(parts[2:])
        full_path = os.path.join(img_root_path, sub_path)

        data_list.append({"image_path": full_path, "caption": cap})
    return Dataset.from_list(data_list)

print("กำลังโหลด Dataset...")
train_ds = load_json_to_dataset(
    "/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/capgen_v1.0_train.json",
    "/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/train/train/"
)

val_ds = load_json_to_dataset(
    "/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/capgen_v1.0_val.json",
    "/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/val/val/"
)
print(f"✅ โหลดสำเร็จ! Train: {len(train_ds)}, Val: {len(val_ds)}")

กำลังโหลด Dataset...
✅ โหลดสำเร็จ! Train: 28004, Val: 4036


In [ ]:
print("กำลังประกอบร่าง Model...")
encoder_id = "google/vit-base-patch16-224"
decoder_id = "flax-community/gpt2-base-thai"

image_processor = AutoImageProcessor.from_pretrained(encoder_id)
tokenizer = AutoTokenizer.from_pretrained(decoder_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

กำลังประกอบร่าง Model...


In [ ]:
encoder = ViTModel.from_pretrained(encoder_id)

decoder_config = AutoConfig.from_pretrained(decoder_id)
decoder_config.is_decoder = True
decoder_config.add_cross_attention = True
decoder_config.max_position_embeddings = 1024

decoder = GPT2LMHeadModel.from_pretrained(decoder_id, config=decoder_config)
decoder.resize_token_embeddings(len(tokenizer))

model = VisionEncoderDecoderModel(encoder=encoder, decoder=decoder)

model.config.decoder_start_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

model.generation_config.max_length = 64
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id
model.generation_config.decoder_start_token_id = tokenizer.eos_token_id

for param in model.encoder.parameters():
    param.requires_grad = False
print("✅ ประกอบร่างสำเร็จและตั้งค่า Generation ถูกต้องแล้ว!")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: flax-community/gpt2-base-thai
Key                                                 | Status     | 
----------------------------------------------------+------------+-
transformer.h.{0...11}.attn.masked_bias             | UNEXPECTED | 
transformer.h.{0...11}.ln_cross_attn.bias           | MISSING    | 
transformer.h.{0...11}.ln_cross_attn.weight         | MISSING    | 
transformer.h.{0...11}.crossattention.c_proj.weight | MISSING    | 
transformer.h.{0...11}.crossattention.c_attn.bias   | MISSING    | 
transformer.h.{0...11}.crossattention.q_attn.bias   | MISSING    | 
transformer.h.{0...11}.crossattention.c_attn.weight | MISSING    | 
transformer.h.{0...11}.crossattention.c_proj.bias   | MISSING    | 
transformer.h.{0...11}.crossattention.q_attn.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from

✅ ประกอบร่างสำเร็จและตั้งค่า Generation ถูกต้องแล้ว!


In [ ]:
def collate_fn(batch):
    images = []
    texts = []

    for item in batch:
        try:
            img = Image.open(item["image_path"]).convert("RGB")
            images.append(img)
            texts.append(item["caption"])
        except:
            continue

    if len(images) == 0:
        images = [Image.new('RGB', (224, 224), color='black')]
        texts = ["ภาพถ่าย"]

    pixel_values = image_processor(images=images, return_tensors="pt").pixel_values

    encodings = tokenizer(
        texts,
        padding="max_length",
        max_length=64,
        truncation=True,
        return_tensors="pt"
    )

    labels = encodings.input_ids
    attention_mask = encodings.attention_mask

    labels[labels >= tokenizer.vocab_size] = tokenizer.eos_token_id
    labels[labels == tokenizer.pad_token_id] = -100

    return {
        "pixel_values": pixel_values,
        "labels": labels,
        "attention_mask": attention_mask
    }

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./vit-gpt2-thai-turbo",

    per_device_train_batch_size=32,
    gradient_accumulation_steps=1,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,

    optim="adamw_torch_fused",
    learning_rate=5e-5,
    num_train_epochs=3,

    fp16=True,
    remove_unused_columns=False,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=50,

    predict_with_generate=True,
    weight_decay=0.01,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=collate_fn
)

print("⚡ เริ่มรัน Turbo Training...")
trainer.train()

⚡ เริ่มรัน Turbo Training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,2.998079
100,2.979976
150,2.916277
200,2.824940
250,2.781515
300,2.736935
350,2.744220
400,2.662697
450,2.681339
500,2.549284


TrainOutput(global_step=1314, training_loss=2.586797737458344, metrics={'train_runtime': 1925.3587, 'train_samples_per_second': 43.634, 'train_steps_per_second': 0.682, 'total_flos': 1.5161132420784194e+19, 'train_loss': 2.586797737458344, 'epoch': 3.0})

In [ ]:
from torch.utils.data import Dataset, DataLoader

print("\n🔍 เริ่มสร้างไฟล์ Submission.csv (Turbo Mode)...")
model.eval()

sample_csv = "/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/sample_submission.csv"
test_dir = "/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning/test/test"
df_sample = pd.read_csv(sample_csv, dtype={'image_id': str})

class TestDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df
        self.img_dir = img_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx]['image_id'])
        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new('RGB', (224, 224), color='black')
        return {"image": image, "img_id": img_id}

def collate_test(batch):
    images = [item["image"] for item in batch]
    img_ids = [item["img_id"] for item in batch]
    # แปลงกลุ่มรูปภาพเป็น pixel_values ทีเดียว
    pixel_values = image_processor(images=images, return_tensors="pt").pixel_values
    return pixel_values, img_ids

test_dataset = TestDataset(df_sample, test_dir)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_test
)

results = []

for pixel_values, img_ids in tqdm(test_loader):
    pixel_values = pixel_values.to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            pixel_values,
            max_length=64,
            num_beams=3,
            no_repeat_ngram_size=2
        )

    captions = tokenizer.batch_decode(output_ids, skip_special_tokens=True)

    for img_id, caption in zip(img_ids, captions):
        if caption.strip() == "":
            caption = "ภาพถ่าย"
        results.append({"image_id": img_id, "caption": caption.strip()})

pd.DataFrame(results).to_csv("submission_turbo.csv", index=False, encoding='utf-8-sig')
print("✅ เสร็จสมบูรณ์! ไฟล์ submission_turbo.csv พร้อมส่งแล้วครับ 🎉")


🔍 เริ่มสร้างไฟล์ Submission.csv (Turbo Mode)...


100%|██████████| 63/63 [04:38<00:00,  4.42s/it]

✅ เสร็จสมบูรณ์! ไฟล์ submission_turbo.csv พร้อมส่งแล้วครับ 🎉


In [ ]:
pd.DataFrame(results).to_csv("submission_last_final.csv", index=False, encoding='utf-8-sig')
print("✅ เสร็จสมบูรณ์! ไฟล์ submission_last_final.csv พร้อมส่งแล้วครับ 🎉")

✅ เสร็จสมบูรณ์! ไฟล์ submission_last_final.csv พร้อมส่งแล้วครับ 🎉
